> **Multi-node workspace required:** This is a multi-node example. To run it,
> your Modal workspace must have multi-node enabled. Contact
> [support@modal.com](mailto:support@modal.com) to enable multi-node.

# Frontier-scale training

When you're decided that you need to train a frontier-scale model
for your workload, you're probably looking for 1) the beefiest
hardware you can get, and 2) guarantees that your runs won't crash.
This tutorial trains [GLM-4.7](https://huggingface.co/zai-org/GLM-4.7)
across 8 nodes with 8 H200s each for a total of 64 GPUs using full-weight
[Group Sequence Policy Optimization](https://arxiv.org/abs/2507.18071) (GSPO)
on the
[zhuzilin/dapo-math-17k](https://huggingface.co/datasets/zhuzilin/dapo-math-17k)
Huggingface dataset. And what do you know: it's easy as pie (possibly even easier).

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from modal_training_gym import (
    Endpoint,
    GLM_4_7,
    HuggingFaceDataset,
    TrainConfig,
    list_checkpoints,
)
from modal_training_gym.train_recipes.slime_recipe import GLM_4_7_Recipe

## Set up

We'll quickly set up the model, dataset, and training recipe. All that you successfully
train this model is encapsulated in the model and recipe classes. However, we surface
some important parameters to make the run correct and fast.

In [ ]:
model = GLM_4_7()

class MathDataset(HuggingFaceDataset):
    hf_repo = "zhuzilin/dapo-math-17k"
    input_key = "prompt"
    label_key = "label"
    output_format = "jsonl"
    apply_chat_template = True
    always_prepare = True

train_dataset = MathDataset(hf_split="train[:2000]")
recipe = GLM_4_7_Recipe(
    rm_type="deepscaler",
)

print(f"training and rollout gpus colocated: {recipe.colocate}")
print(f"training nodes: {recipe.actor_num_nodes}, gpus/node: {recipe.actor_num_gpus_per_node}")
print(f"rollout gpus: {recipe.rollout_num_gpus}, rollout gpus/engine: {recipe.rollout_num_gpus_per_engine}")
print(f"parallelism: tp={recipe.tensor_model_parallel_size}, pp={recipe.pipeline_model_parallel_size}, "
      f"cp={recipe.context_parallel_size}, ep={recipe.expert_model_parallel_size}")
print(f"optimizer cpu offload: {recipe.optimizer_cpu_offload}")

## Kick off training

Let's begin, shall we?

In [ ]:
config = TrainConfig(
    model=model,
    dataset=train_dataset,
    recipe=recipe,
)

run = config.launch()
print(f"run id: {train_result.training_run_id}")

## Test out the trained model

We'll spin up an [Endpoint](https://modal.com/docs/guide/endpoints)
and see how the trained model does.

In [ ]:
result = run.result()
checkpoint = list_checkpoints(result.training_run_id)[-1]
print(f"checkpoint: {checkpoint.path}")

trained_deployment = Endpoint.launch(
    model, checkpoint, unauthenticated=True, recreate_if_existing=True
)
deployment.wait_until_ready(timeout=45 * 60)
print(f"checkpoint deployed to {trained_deployment.url}")

msg = deployment.chat(
    [
        {
            "role": "user",
            "content": (
                "Let $p$ be a prime number. Find the number of integers $n$ "
                "with $1 \\le n \\le p^2$ such that $n^{p-1} \\equiv 1 \\pmod{p^2}$."
            ),
        }
    ],
)
print(msg.get("content") or msg.get("reasoning_content") or "")